# Supplementary Figure 3 — task, novelty, SWR rate and avalanches

Panel a: T-maze performance (Mouse 1–6); b–d on non-overlapping windows (b novelty, c SWR rate, d the d2-uncertainty shade plot); e the 1× IEI avalanche example (sizes and durations); f DCC and KS at the 30th percentile; g DCC by SWR-rate tercile. Every panel is drawn from its producer's table or notebook rather than recomputed here. d2 = d2 (2024 Euclidean, AR p = 20, 40 ms), labelled "d2". Session counts are in the figure legend, not on the panels.

- **a** T-maze alternation, fraction correct per session (all T-maze trials of the day), dots coloured by mouse (Fig. 1d's colours); `m25_tmaze_box.py` → `m25_tmaze_sessions.csv`: 17 sessions, 6 mice; one-sided signed-rank vs chance (0.5).
- **b** Novelty, one panel (the novel-vs-control Δ): per session, d2 normalised by baseline (d2_rel = (d2 − m_pre)/(d2 + m_pre), m_pre = pre-task sleep; median per phase), Δ = post − mid sleep; on a novel day mid-sleep follows a familiar task and post-sleep the novel one, on a control day both follow familiar tasks. Non-overlapping windows (`ed_nonoverlap_tables/d2_rel_novel_familiar.csv`, PPP8_day14 from its own d2), 8 control / 9 novel sessions. Mixed model of Δ on session × novelty (animal and session grouping; the per-session fit is singular, so the animal grouping is used), checked against an exact permutation test. Above it the submitted ED2a schematic, cut unchanged (typo "Familar" fixed in the copy); colours as submitted (familiar #dd55ff, novel #8800aa). y label "Δ d2 (normalized by baseline)"; Δ = post − mid sleep is stated in the legend.
- **c** SWR rate, presleep vs postsleep (the mid-session sleep, as Fig. 2a): the Fig. 2a notebook's own per-session table on non-overlapping windows (`fig2a_prepost_sessions_disjoint.csv`), 9 sessions; its mixed model on the session means (1|animal) + (1|animal_day). Submitted colour #b04d4e.
- **d** d2 (z) and its uncertainty by SWR-rate sextile: the submitted ED2c notebook on non-overlapping windows (tiles and z recomputed); shade = d2 uncertainty, error bars = SD; linear trend fit to the six sextile means.
- **e** Avalanche example, PPP20_day4 NREM, 1× IEI (1.93 ms bins), 30th percentile, low vs high local SWR rate (median split): size and duration CCDFs with the common-xmin power-law fits (dashed), and mean size vs duration. The batch notebook's `run_session` on that session, reproduced to the stored `ripple_criticality_PPP20_day4.csv` to numerical precision, cached as `data/revision/log/figs3e_example_PPP20_day4_avalanches.npz`. No exponents are printed on the panel.
- **f** 30th percentile, 16 sessions, low vs high SWR rate, paired signed-rank: DCC (common xmin; `ripple_criticality_all_sessions.csv`); KS distance of the sizes (xmin per group; `ks_xmin_convention.csv`, `ks_free`). Same table as `m22_dcc_threshold_box`, 30th-percentile column only.
- **g** DCC by SWR-rate tercile (same table, 30th percentile; median and IQR over sessions, grey = sessions; `fig2_new.panel_dcc`); high vs low paired signed-rank.
- Log-axis exponents (10⁻²) print at 3.8 pt (matplotlib's superscript scale); everything else is 5–6 pt.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('.'))   # notebooks/figures_revision
import final_figs as F                     # paths, save/copy helpers; adds the revision folder to sys.path

## a — T-maze performance

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import m25_tmaze_box as M25

# a: T-maze alternation performance, one dot per session coloured by mouse (Mouse 1-6, the colours of Fig. 1d), from
# m25_tmaze_box.py's table (data/revision/log/m25_tmaze_sessions.csv: all T-maze trials of the day per session).
# Statistic as the producer's: one-sided signed-rank of the session fractions against chance (0.5).
S = pd.read_csv(M25.OUT_CSV)
w = stats.wilcoxon(S.frac_correct - 0.5, alternative="greater")
print(f"a: fraction correct vs chance (0.5), one-sided Wilcoxon signed-rank p = {w.pvalue:.2g}; n = {len(S)} sessions "
      f"from {S.animal.nunique()} mice")
F.final_style()
fig, ax = plt.subplots(figsize=(32 * F.MM, 50 * F.MM), layout="constrained")
ax.boxplot(S.frac_correct, positions=[0], widths=0.5, showfliers=False, patch_artist=True,
           boxprops=dict(facecolor="0.9", edgecolor="0.3", lw=0.6), medianprops=dict(color="k", lw=0.9),
           whiskerprops=dict(lw=0.6), capprops=dict(lw=0.6))
rng = np.random.default_rng(1)                         # the producer's jitter
for a in sorted(S.animal.unique(), key=lambda x: M25.MOUSE_LABEL[x]):
    g = S[S.animal == a]
    ax.scatter(rng.uniform(-0.15, 0.15, len(g)), g.frac_correct, s=7, color=M25.MOUSE_COL.get(a, "0.5"),
               edgecolor="white", lw=0.3, zorder=3, label=M25.MOUSE_LABEL[a])
ax.axhline(0.5, color="0.4", ls="--", lw=0.5)
ax.text(0.47, 0.505, "chance", fontsize=5, color="0.4", va="bottom", ha="right")
ax.text(0, 0.985, F.stars(w.pvalue), ha="center", va="top")
ax.set_xlim(-0.5, 0.5); ax.set_ylim(0.4, 1.0); ax.set_xticks([])
ax.set_ylabel("T-maze alternation\n(fraction correct)")
ax.legend(frameon=False, fontsize=5, loc="lower left", ncol=2, handletextpad=0.05, columnspacing=0.4,
          borderaxespad=0.1, labelspacing=0.25)
F.show(F.save(fig, "FigS3_A_tmaze_performance"))

## b — novelty: novel vs control Δ, with the submitted schematic

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from ed2a_novel_vs_control import per_session, exact_perm, lmm_interaction, LOGDIR

# b: novelty, as one panel -- control vs novel, each box the per-session change from mid- to post-task sleep of d2
# normalised by baseline; y label "d2 (normalized by baseline)"; the submitted ED2a schematic is shown above it,
# with its original composition and colours.
# d2 = d2 (2024 Euclidean, p = 20, 40 ms); d2_rel = (d2 - m_pre) / (d2 + m_pre), m_pre = the session's pre-task sleep;
# per session the median d2_rel per phase; Delta = post - mid. On a novel day mid-sleep follows a familiar task and
# post-sleep the novel one; on a control day both follow familiar tasks. Panels b-d use non-overlapping windows:
# ed_nonoverlap_tables/d2_rel_novel_familiar.csv (PPP8_day14 from its own d2).

# ---- the submitted schematic (Extended Data Fig. 2a; mm units), cut out unchanged, its letter left out
ED2 = os.path.join(F.REPO, "reports", "new_planc", "nature", "Extended Data Fig2.svg")
F.crop_svg(ED2, "FigS3_B_schematic_novel_familiar", 4.2, 5.6, 36.0, 11.5, drop_ids=["text910277-5-8"], unit_mm=1.0)
_sch = F.stem("FigS3_B_schematic_novel_familiar") + ".svg"          # the label "Familar" set to "Familiar" in the copy
_t = open(_sch, encoding="utf-8").read(); assert _t.count(">Familar<") == 1
open(_sch, "w", encoding="utf-8").write(_t.replace(">Familar<", ">Familiar<"))
import subprocess; subprocess.run([F.INKSCAPE, _sch, "--export-type=png", "--export-dpi=600", "--export-background=#ffffff",
                                  f"--export-filename={_sch[:-4]}.png"], capture_output=True)
F.show(F.stem("FigS3_B_schematic_novel_familiar") + ".png")

# ---- the statistic: the difference in differences, tested with the paper's mixed model (session as the unit)
s = per_session(pd.read_csv(os.path.join(LOGDIR, "ed_nonoverlap_tables", "d2_rel_novel_familiar.csv"),
                            float_precision="round_trip"), "median")
obs, p_perm, n_perm = exact_perm(s.delta.to_numpy(), s.novel.to_numpy())
lmm = lmm_interaction(s)
p_lmm = lmm.get("lmm_file_p", lmm.get("lmm_animal_p"))          # per-file fit is singular here -> animal grouping
print(f"{int((s.novel == 0).sum())} control, {int((s.novel == 1).sum())} novel sessions; novel - control Delta = {obs:+.4f}; "
      f"mixed model (S2 x novel, {'file' if 'lmm_file_p' in lmm else 'animal + file'}) p = {p_lmm:.3g}; "
      f"exact permutation p = {p_perm:.3g} ({n_perm} assignments)")

# ---- the panel, in the submitted panel's colours: familiar (control) #dd55ff, novel #8800aa
COL = {"control": "#dd55ff", "novel": "#8800aa"}
F.final_style()
fig, ax = plt.subplots(figsize=(36 * F.MM, 42 * F.MM), layout="constrained")
groups = [("control", s[s.novel == 0].delta.to_numpy()), ("novel", s[s.novel == 1].delta.to_numpy())]
bp = ax.boxplot([g for _, g in groups], positions=[0, 1], widths=0.55, patch_artist=True, showfliers=False,
                medianprops=dict(color="k", lw=0.8), boxprops=dict(lw=0.6), whiskerprops=dict(lw=0.6), capprops=dict(lw=0.6))
for patch, (name, _) in zip(bp["boxes"], groups):
    patch.set_facecolor(COL[name])
rng = np.random.default_rng(0)
for i, (_, g) in enumerate(groups):
    ax.scatter(i + rng.uniform(-0.09, 0.09, g.size), g, s=5, color="k", alpha=0.8, lw=0, zorder=3)
ax.axhline(0, ls="--", lw=0.5, color="gray")
ax.set_xticks([0, 1]); ax.set_xticklabels(["Familiar\n(control)", "Novel"]); ax.set_xlim(-0.6, 1.6)
ax.set_ylabel("Δ d2 (normalized by baseline)")   # Δ = post − mid sleep: stated in the legend
lo, hi = ax.get_ylim(); span = hi - lo
F.bracket(ax, 0, 1, hi + 0.04 * span, F.stars(p_lmm), h=0.02 * span)
ax.set_ylim(lo, hi + 0.14 * span)
F.settle(fig)
F.drop_counts(fig)
F.show(F.save(fig, "FigS3_B_novel_vs_control"))

## c, d — SWR rate presleep vs postsleep; d2 uncertainty by SWR rate (non-overlapping)

In [ ]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# c: SWR rate, presleep vs postsleep -- the submitted Extended Data Fig. 2b (the ripple-rate half of the Fig. 2a
# notebook) on non-overlapping windows: the notebook's own per-session table (fig2a_prepost_sessions_disjoint.csv,
# NREM, presleep vs midsleep -- labelled postsleep -- per-session means, 9 sessions), its statistic the notebook's
# mixed model on the session means (1|animal) + (1|animal_day), read from that run's log. Submitted colour #b04d4e.
T = pd.read_csv(os.path.join(F.REPO, "data", "revision", "log", "fig2a_prepost_sessions_disjoint.csv"))
P = T.pivot(index="animal_day", columns="epoch2", values="ripple_rate_30s").dropna()[["Presleep", "Postsleep"]]
log = open(os.path.join(F.FIGS, "ed_nonoverlap", "Fig2_A_FigS2_B_FigS3_A.log"), encoding="utf-8").read()
m = re.search(r"Effect \(Postsleep – Presleep\) on ripple_rate_30s:\s*Δ = ([0-9.eE+-]+).*?p = ([0-9.eE+-]+)", log, re.S)
dlt_c, p_c = float(m.group(1)), float(m.group(2))
assert abs(dlt_c - (P.Postsleep - P.Presleep).mean()) < 0.02, "the log is not this table"
print(f"c: SWR rate post vs pre, paired linear mixed model p = {p_c:.3g}; n = {len(P)} sessions")
F.final_style()
fig, ax = plt.subplots(figsize=(30 * F.MM, 50 * F.MM), layout="constrained")
L = P.reset_index().melt(id_vars="animal_day", var_name="epoch2", value_name="rate")
order = ["Presleep", "Postsleep"]
sns.boxplot(data=L, x="epoch2", y="rate", order=order, color="#b04d4e", showfliers=False, width=0.6, linewidth=0.6,
            ax=ax, zorder=2)
for _, r in P.iterrows():
    ax.plot([0, 1], [r.Presleep, r.Postsleep], color="gray", alpha=0.6, lw=0.5, zorder=3)
ax.scatter(np.r_[np.zeros(len(P)), np.ones(len(P))], np.r_[P.Presleep, P.Postsleep], s=5, color="k", zorder=4, lw=0)
lo, hi = ax.get_ylim(); span = hi - lo
F.bracket(ax, 0, 1, hi + 0.03 * span, F.stars(p_c), h=0.02 * span)
ax.set_ylim(lo, hi + 0.13 * span)
ax.set_xlabel(""); ax.set_ylabel("SWR rate (Hz)")
F.settle(fig)
F.drop_counts(fig)
F.show(F.save(fig, "FigS3_C_swr_rate_prepost"))

# d: d2 and its uncertainty by SWR-rate sextile -- the submitted Extended Data Fig. 2c notebook (FigS1_C_SNR_S2C_d2
# uncertainty) on non-overlapping windows (ed_nonoverlap_tables/d2_err_by_swr_rate_sextile.csv: tiles and z
# recomputed on the disjoint windows), its figure captured and restyled; its statistic the notebook's linear trend of
# the six pooled means, read from the run's log.
CAPD = F.run_rv1_capture("FigS1_C_SNR_S2C_d2 uncertainty", ["S2_C_ripple_rate_by_d2_tile_pooled_std"])
log = open(F.LAST_RV1_LOG, encoding="utf-8").read()
m = re.search(r"Linear trend on pooled means.*?R² = ([0-9.]+), p = ([0-9.eE+-]+)", log, re.S)
r2_d, p_d = float(m.group(1)), float(m.group(2))
print(f"d: linear regression of the six sextile means, R² = {r2_d:.2f}, p = {p_d:.2g}")
fig = CAPD["S2_C_ripple_rate_by_d2_tile_pooled_std"]; ax = fig.axes[0]
ax.set_ylabel("d2 (z)"); ax.set_xlabel("SWR-rate sextile")
F.restyle(fig, 44, 50)
ax.text(0.5, 0.97, F.stars(p_d), transform=ax.transAxes, ha="center", va="top", fontsize=6)
F.show(F.save(fig, "FigS3_D_d2_uncertainty_by_swr_rate"))

## e–g — avalanches, 1× IEI, NREM only

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from mixture_proof import fit_powerlaw, ccdf_at
import m22_dcc_threshold_box as M22
import fig2_new as N

# e-g: neuronal avalanches, from the avalanche pipeline (ripple_criticality_batch.ipynb; bin = 1x the population's
# mean inter-event interval, detection strictly inside NREM, 30th-percentile activity threshold, common xmin; each
# avalanche labelled by the local SWR rate, 10 s window, median split low / high, terciles for g). No exponents are
# shown on the panels; KS distance and DCC are reported instead. Low / high colours as m22_dcc_threshold_box's (f).
C = M22.C                                              # {"LOW": blue, "HIGH": red}

# ---- e: the example session PPP20_day4. Its avalanches are the batch notebook's run_session (cells 1, 3) on that
# session -- reproduced to the stored ripple_criticality_PPP20_day4.csv within floating-point precision and cached
# as an npz.
EX = os.path.join(F.REPO, "data", "revision", "log", "figs3e_example_PPP20_day4_avalanches.npz")
if not os.path.exists(EX):
    nb = json.load(open(os.path.join(F.REV, "ripple_criticality_batch.ipynb"), encoding="utf-8"))
    code = {i: "".join(c["source"]) for i, c in enumerate(nb["cells"]) if c["cell_type"] == "code"}
    _ns = {"__name__": "figs3e"}
    for i in (1, 3):
        exec(compile(code[i], f"batch:cell{i}", "exec"), _ns)
    _df, raw = _ns["run_session"](r"W:\data\PPP\PPP20\day4", "PPP20_day4", verbose=False)    # W: read-only
    np.savez_compressed(F.assert_local_path(EX), **{k: raw[k] for k in ("S", "D", "rr", "med", "xmin_s", "xmin_d", "mean_iei")})
Z = np.load(EX)
S, D, rr, med = Z["S"], Z["D"], Z["rr"], float(Z["med"])
xmin_s, xmin_d, iei_ms = int(Z["xmin_s"]), int(Z["xmin_d"]), 1000 * float(Z["mean_iei"])
lo, hi = rr <= med, rr > med
print(f"e: PPP20_day4, {len(S)} NREM avalanches, bin {iei_ms:.2f} ms, median local SWR rate {med:.2f} Hz; "
      f"common xmin size {xmin_s}, duration {xmin_d}")
F.final_style()
fig, axes = plt.subplots(1, 3, figsize=(100 * F.MM, 36 * F.MM), layout="constrained")   # wide and short layout
for ax, X, xmin, lab in ((axes[0], S, xmin_s, "Avalanche size"), (axes[1], D, xmin_d, f"Duration ({iei_ms:.1f} ms bins)")):
    grid = np.unique(np.round(np.logspace(0, np.log10(X.max()), 60)))
    for mask, key, name in ((lo, "LOW", "Low SWR rate"), (hi, "HIGH", "High SWR rate")):
        x = X[mask]; x = x[x > 0].astype(float)
        ax.loglog(grid, ccdf_at(x, grid), "o", ms=1.4, mew=0, color=C[key], label=name)
        a, _ = fit_powerlaw(x[x >= xmin], xmin, int(X.max()))          # common xmin, as the pipeline's fits
        th = np.logspace(np.log10(xmin), np.log10(X.max()), 50)
        ax.loglog(th, ccdf_at(x, np.array([xmin]))[0] * (th / xmin) ** (-(a - 1)), "--", color=C[key], lw=0.6)
    ax.set_xlabel(lab); ax.set_ylabel("CCDF")
axes[0].legend(frameon=False, loc="lower left", fontsize=5, handletextpad=0.1, borderaxespad=0.1)
ax = axes[2]
for mask, key in ((lo, "LOW"), (hi, "HIGH")):
    dd, ss = D[mask], S[mask]
    ud = np.unique(dd[dd >= xmin_d])
    ms_ = np.array([ss[dd == t].mean() for t in ud])
    ok = ms_ > 0
    ax.loglog(ud[ok], ms_[ok], "o", ms=1.4, mew=0, color=C[key])
ax.set_xlabel(f"Duration ({iei_ms:.1f} ms bins)"); ax.set_ylabel("Mean size")
F.show(F.save(fig, "FigS3_E_avalanche_example"))

# ---- f: DCC and KS distance, low vs high SWR rate, 30th percentile only; m22_dcc_threshold_box's tables: DCC from
# ripple_criticality_all_sessions.csv (common xmin), KS of the sizes from ks_xmin_convention.csv (bin_mult 1, xmin
# per group, ks_free); paired signed-rank over the 16 sessions, as in m22_dcc_threshold_box.
d = pd.read_csv(os.path.join(M22.LOG, M22.SRC[30]))
DCC = d[d.group.isin(["LOW", "HIGH"])].pivot_table(index="session", columns="group", values="DCC").dropna()
k = pd.read_csv(os.path.join(M22.LOG, M22.KS)); k = k[(k.bin_mult == 1) & (k.pct == 30)]
KS = k.pivot_table(index="session", columns="group", values="ks_free").dropna()
fig, axes = plt.subplots(1, 2, figsize=(43 * F.MM, 46 * F.MM), layout="constrained")
for ax, P, ylab, log in ((axes[0], DCC, "DCC", True), (axes[1], KS, "KS distance D", False)):   # "(avalanche sizes)" is given in the legend
    for j, grp in enumerate(("LOW", "HIGH")):
        ax.boxplot(P[grp], positions=[j], widths=0.55, showfliers=False, patch_artist=True,
                   boxprops=dict(facecolor=C[grp], alpha=0.35, edgecolor=C[grp], lw=0.6),
                   medianprops=dict(color=C[grp], lw=1.0), whiskerprops=dict(color=C[grp], lw=0.6),
                   capprops=dict(color=C[grp], lw=0.6))
    for _, r in P.iterrows():
        ax.plot([0, 1], [r.LOW, r.HIGH], "-", color="#999999", lw=0.4, zorder=2)
    ax.scatter(np.r_[np.zeros(len(P)), np.ones(len(P))], np.r_[P.LOW, P.HIGH], s=4, lw=0, zorder=3,
               color=[C["LOW"]] * len(P) + [C["HIGH"]] * len(P))
    if log:
        ax.set_yscale("log")
    pw = stats.wilcoxon(P.HIGH, P.LOW).pvalue
    print(f"f {ylab.splitlines()[0]}: high vs low SWR rate, two-sided Wilcoxon signed-rank p = {pw:.2g}; "
          f"n = {len(P)} sessions")
    lo_, hi_ = ax.get_ylim()
    if log:
        y = hi_ * 1.4
        ax.plot([0, 0, 1, 1], [y / 1.15, y, y, y / 1.15], color="k", lw=0.5)
        ax.text(0.5, y * 1.05, F.stars(pw), ha="center", va="bottom"); ax.set_ylim(lo_, y * 3)
    else:
        span = hi_ - lo_
        F.bracket(ax, 0, 1, hi_ + 0.03 * span, F.stars(pw), h=0.02 * span); ax.set_ylim(lo_, hi_ + 0.13 * span)
    ax.set_xticks([0, 1]); ax.set_xticklabels(["Low", "High"]); ax.set_xlim(-0.6, 1.6)
    ax.set_xlabel("SWR rate"); ax.set_ylabel(ylab)
    if not log:                                          # KS: a tick every 0.01, labelled 0.01, 0.02 ...
        from matplotlib.ticker import MultipleLocator, FormatStrFormatter
        ax.yaxis.set_major_locator(MultipleLocator(0.01)); ax.yaxis.set_major_formatter(FormatStrFormatter("%.2f"))
F.show(F.save(fig, "FigS3_F_dcc_ks_low_high"))

# ---- g: DCC by SWR-rate tercile (fig2_new.panel_dcc: the same table, 30th percentile, groups T1-T3; median and IQR
# over sessions, grey lines = sessions; high vs low signed-rank), drawn by the producer and restyled
F.native_style()
fig, ax = plt.subplots(figsize=(34 * F.MM, 46 * F.MM), layout="constrained")
N.panel_dcc(ax)
ax.set_xlabel("SWR-rate tercile (NREM)"); ax.set_ylabel("DCC")
F.shrink(fig)
t = [t for t in ax.texts if t.get_text().startswith("high > low")][0]      # the producer's "high > low N/M *" text, redrawn as a bracket
lab = t.get_text().replace("high > low", "").strip(); t.remove()
y = max(l.get_ydata().max() for l in ax.lines if len(l.get_ydata())) * 1.5
ax.plot([1, 1, 3, 3], [y / 1.12, y, y, y / 1.12], color="k", lw=0.5, clip_on=False)
ax.text(2, y * 1.05, lab, ha="center", va="bottom", fontsize=6)
ax.set_ylim(ax.get_ylim()[0], y * 2.4)
F.settle(fig)
F.drop_counts(fig)
F.show(F.save(fig, "FigS3_G_dcc_by_swr_tercile"))